# Session 6 lab · Local features & matching

The code behind the slides. Each cell shows **one idea** in a few lines:

1. run it and look at the result;
2. change a value marked **`· try`** (the values after it are good ones to try) and run it again;
3. read *What to notice* under the cell.

| Part | Ideas |
|---|---|
| **A · Corners** | the change of a sliding window · M from gradients · Taylor and the eigenvalues, measured · Harris against `cv2.cornerHarris` |
| **B · Scale** | zoom turns a corner into an edge · scale space · a blob answers at its own size · SIFT on a shrunk copy |
| **C · Descriptors & matching** | 128 numbers per point · nearest descriptor and the ratio test, judged right or wrong |
| **D · RANSAC** | one wrong point ruins least squares · RANSAC by hand · how many tries · a homography for the photo pair · a second view with a known answer |

Nothing to hand in, nothing to upload — it runs on free Colab with OpenCV only. The slides' numbers come from OpenCV 5.0; another version can shift a count by a little.

In [ ]:
# Setup — run this cell first
import numpy as np, matplotlib.pyplot as plt, cv2, urllib.request, os

def photo(name):                                   # the two photos from the slides (CC0, Wikimedia Commons)
    if not os.path.exists(name):
        urllib.request.urlretrieve('https://raw.githubusercontent.com/INCORTX/INCORTX.github.io/cv-next/ComputerVision/session-06/img/' + name, name)
    return cv2.imread(name)

full, detail = photo('full.jpg'), photo('detail.jpg')          # the whole building, and a close-up of it
gA, gB = cv2.cvtColor(full, cv2.COLOR_BGR2GRAY), cv2.cvtColor(detail, cv2.COLOR_BGR2GRAY)
s = 400 / max(gA.shape)                                        # part A works on the photo at 400 px, like the slide demos
G = cv2.resize(gA, None, fx=s, fy=s, interpolation=cv2.INTER_AREA).astype(float) / 255
POINTS = {'flat': (191, 39), 'edge': (269, 65), 'corner': (301, 209)}           # (x, y) of the flat / edge / corner windows of slide 5
sift = cv2.SIFT_create()
kA, dA = sift.detectAndCompute(gA, None); kB, dB = sift.detectAndCompute(gB, None)
good = [m for m, n in cv2.BFMatcher().knnMatch(dB, dA, k=2) if m.distance < 0.8 * n.distance]
H_REF = cv2.findHomography(np.float32([kB[m.queryIdx].pt for m in good]), np.float32([kA[m.trainIdx].pt for m in good]), cv2.RANSAC, 4.0)[0]   # close-up → whole; part D shows how — part C uses it to judge a match
print('OpenCV', cv2.__version__, '· whole', gA.shape[::-1], f'{len(kA):,} SIFT points · close-up', gB.shape[::-1], f'{len(kB):,} points')

## A · Corners

Slide a small window: a **corner** changes whichever way it moves, an **edge** only across, a **flat** patch not at all. The gradients in the window make a 2 × 2 matrix **M**; its eigenvalues say which kind.

In [ ]:
# A1 · Shift a 15 × 15 window and measure the change (slide 5)
where = 'corner'     # · try 'edge', 'flat'

x, y = POINTS[where]; r = 7
win = G[y - r:y + r + 1, x - r:x + r + 1]
E = np.array([[((G[y - r + v:y + r + 1 + v, x - r + u:x + r + 1 + u] - win) ** 2).sum() for u in range(-6, 7)] for v in range(-6, 7)])
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
ax[0].imshow(G, cmap='gray'); ax[0].add_patch(plt.Rectangle((x - r, y - r), 15, 15, fill=False, color='red', lw=2)); ax[0].set_title(where)
ax[1].imshow(E, cmap='magma', extent=(-6.5, 6.5, 6.5, -6.5)); ax[1].set_title('change E for each shift (u, v)'); plt.show()
print(f'{where}: E for a shift of 1 right {E[6, 7]:.2f} · 1 down {E[7, 6]:.2f} · largest {E.max():.2f}')

**What to notice:** the corner's map is dark only at (0, 0): one pixel right changes the window by 6.33, one down by 8.25 — every direction changes it, like the corner button on slide 5. 'edge' changes 13.18 across but 5.72 along, so a darker band runs along the edge; 'flat' stays near 0 everywhere (largest 0.02).

In [ ]:
# A2 · M from the gradients in the window (slide 7) — the deck's recipe
sigma = 1.5     # the Gaussian window · try 3

SX = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float)               # Sobel (session 3 C)
gx = cv2.filter2D(G, -1, SX, borderType=cv2.BORDER_REPLICATE); gy = cv2.filter2D(G, -1, SX.T, borderType=cv2.BORDER_REPLICATE)
n = 2 * round(2 * sigma) + 1; g1 = cv2.getGaussianKernel(n, sigma); GW = g1 @ g1.T
Sxx, Sxy, Syy = (cv2.filter2D(a, -1, GW, borderType=cv2.BORDER_REPLICATE) for a in (gx * gx, gx * gy, gy * gy))
for name, (x, y) in POINTS.items():
    M = np.array([[Sxx[y, x], Sxy[y, x]], [Sxy[y, x], Syy[y, x]]]); l2, l1 = np.linalg.eigvalsh(M)
    print(f'{name:6s} M = [{M[0, 0]:5.2f} {M[0, 1]:5.2f} / {M[1, 0]:5.2f} {M[1, 1]:5.2f}]   λ₁ {l1:.2f}  λ₂ {l2:.2f}')

**What to notice:** these are slide 7's matrices — edge [5.63 −1.13 / −1.13 0.25] with λ₁ 5.86, λ₂ 0.02; corner [2.06 −0.27 / −0.27 1.81] with λ₁ 2.23, λ₂ 1.64, the dots of slide 9. σ = 3 averages over a wider window: the corner's eigenvalues drop to 1.73 and 1.20, and the edge's λ₂ grows to 0.25 because the window now reaches past the edge. A3 and A4 use the window you set here.

In [ ]:
# A3 · Taylor, measured: shifting by u changes the window by about uᵀMu (slides 6 and 8)
where = 'corner'   # · try 'edge'
blur = 2           # σ of a blur first — Taylor needs a picture that is smooth over one pixel · try 0, 4

Gs = cv2.GaussianBlur(G, (0, 0), blur) if blur else G
gy_, gx_ = np.gradient(Gs)                                  # plain slopes, per pixel
x, y = POINTS[where]; r = GW.shape[0] // 2; box = (slice(y - r, y + r + 1), slice(x - r, x + r + 1))
M = np.array([[(GW * gx_[box] ** 2).sum(), (GW * gx_[box] * gy_[box]).sum()], [(GW * gx_[box] * gy_[box]).sum(), (GW * gy_[box] ** 2).sum()]])
print('shift      measured E   Taylor uᵀMu   (both × 1000)')
for (u, v), arrow in zip(((1, 0), (1, 1), (0, 1), (-1, 1)), '→↘↓↙'):
    moved = Gs[y - r + v:y + r + 1 + v, x - r + u:x + r + 1 + u]
    print(f'{arrow} ({u:2d}, {v})   {1000 * (GW * (moved - Gs[box]) ** 2).sum():8.2f}   {1000 * np.array([u, v]) @ M @ [u, v]:10.2f}')
l2, l1 = np.linalg.eigvalsh(M); print(f'λ₂ = {1000 * l2:.2f}, λ₁ = {1000 * l1:.2f}: a shift of length 1 changes E by an amount between them')

**What to notice:** with blur 2 the measured change and the Taylor guess agree for → (3.47 vs 3.52) and ↓ (1.08 vs 1.09); the diagonal shifts move √2 px, so the guess is rougher (5.82 vs 4.24). Per unit of length every shift lands between λ₂ = 1.08 and λ₁ = 3.54 (slide 8) — ↓ is almost exactly λ₂. Blur 0: the raw photo changes faster than its slope predicts (corner → 54.73 vs 35.51): Taylor needs a picture that is smooth over the step. Blur 4 (edge): the guesses come within about 0.1 (× 1000), but most of the detail is gone (λ₁ only 0.46).

In [ ]:
# A4 · Harris: R = det M − k·(trace M)², keep local maxima above a threshold (slide 10)
k = 0.04            # · try 0.06, 0.02
threshold = 0.01    # share of the largest R · try 0.05, 0.001

R = Sxx * Syy - Sxy ** 2 - k * (Sxx + Syy) ** 2
inner = np.zeros(R.shape, bool); inner[8:-8, 8:-8] = True
def corners(R):
    return np.argwhere((R >= cv2.dilate(R, np.ones((3, 3)))) & (R > threshold * R[inner].max()) & inner)
ours, cv = corners(R), corners(cv2.cornerHarris(np.float32(G), 7, 3, k))   # OpenCV: a 7 × 7 box window instead of a Gaussian
near = np.mean([np.abs(cv - p).sum(1).min() <= 2 for p in ours])
plt.figure(figsize=(6, 7)); plt.imshow(G, cmap='gray'); plt.scatter(ours[:, 1], ours[:, 0], s=6, c='r'); plt.axis('off'); plt.show()
print(f"k {k}, threshold {threshold}: {len(ours)} corners · cv2.cornerHarris {len(cv)} · {near:.0%} of ours within 2 px of one of OpenCV's")

**What to notice:** 518 corners at k 0.04 and 1% — slide 10's count; threshold 0.05 keeps 332, 0.001 keeps 670. k hardly changes the count (0.02 → 517, 0.06 → 524 at 1%): it decides how edge-like a point may be, and the strong corners pass anyway. OpenCV's cornerHarris finds 794 with its 7 × 7 box window, and about half of ours sit within 2 px of one of its — the window shape changes which points rank highest. (With σ = 3 in A2 the counts fall: fewer, coarser corners.)

## B · Scale

One window size cannot fit every photo: zoom in and a corner becomes an edge. Blur at every σ (**scale space**) and look for the σ where a point answers most — that σ is its size.

In [ ]:
# B1 · Zoom in, and a corner becomes an edge (slide 14)
window = 15     # pixels · try 31

def eig_at(g, x, y, n):
    g = g.astype(float) / 255; gx = cv2.Sobel(g, cv2.CV_64F, 1, 0); gy = cv2.Sobel(g, cv2.CV_64F, 0, 1); r = n // 2
    w = cv2.getGaussianKernel(n, n / 6); w = w @ w.T; b = (slice(y - r, y + r + 1), slice(x - r, x + r + 1))
    M = np.array([[(w * gx[b] ** 2).sum(), (w * gx[b] * gy[b]).sum()], [(w * gx[b] * gy[b]).sum(), (w * gy[b] ** 2).sum()]])
    l2, l1 = np.linalg.eigvalsh(M); return l1, l2, ('corner' if l2 > 0.25 * l1 else 'edge' if l2 < 0.05 * l1 else 'in between')
for name, g, (x, y) in (('whole', gA, (322, 419)), ('close-up', gB, (453, 141))):    # the same spot of the building
    print(f'{name:8s} window {window}: λ₁ %.3f  λ₂ %.3f → %s' % eig_at(g, x, y, window))

**What to notice:** window 15: the spot is a corner in the whole photo (λ 0.940 / 0.439) and an edge in the close-up (1.264 / 0.001) — slide 14. Window 31, about the 2.2× zoom: the close-up reads 'in between' (λ₂ 0.228) — the larger window sees the corner again. Choosing the window to fit the zoom is what scale space does automatically.

In [ ]:
# B2 · Scale space: the same photo at every blur (slide 15)
which = 'whole'     # · try 'close-up'

g = gA if which == 'whole' else gB
fig, ax = plt.subplots(1, 6, figsize=(15, 3.4))
for a, s in zip(ax, (0, 1, 2, 4, 8, 16)):
    a.imshow(g if s == 0 else cv2.GaussianBlur(g, (0, 0), s), cmap='gray'); a.set_title(f'σ = {s}'); a.axis('off')
plt.show()

**What to notice:** bricks go first, then the windows, then whole floors (slide 15). The close-up is 2.2× bigger, so the same details survive about 2.2× more blur.

In [ ]:
# B3 · A blob answers most at its own size: σ²·Laplacian at the centre of a disc (slide 16)
extra = 12      # radius of a fourth disc · try 6, 24

sig = np.arange(1, 24.01, 0.25)
for r in (4, 8, 16, extra):
    disc = np.zeros((241, 241), np.float32); cv2.circle(disc, (120, 120), r, 1.0, -1, lineType=cv2.LINE_AA)
    resp = [-(s ** 2) * cv2.Laplacian(cv2.GaussianBlur(disc, (0, 0), s), cv2.CV_32F)[120, 120] for s in sig]
    plt.plot(sig, resp, label=f'r = {r}'); print(f'r = {r:2d}: peak at σ = {sig[np.argmax(resp)]:5.2f} · r/√2 = {r / 2 ** 0.5:5.2f}')
plt.xlabel('σ'); plt.ylabel('σ² · −Laplacian at the centre'); plt.legend(); plt.show()

**What to notice:** the discs peak at σ 2.75 / 5.50 / 11.25 for r 4 / 8 / 16 — slide 16 — next to r/√2 = 2.83 / 5.66 / 11.31. The extra disc follows the same rule: r 12 → 8.50 (8.49), r 6 → 4.25 (4.24), r 24 → 17.00 (16.97). The peak gives the size.

In [ ]:
# B4 · SIFT on the photo and on a shrunk copy: the same points, at a smaller size (slide 17)
scale = 0.5     # · try 0.25, 0.75

small = cv2.resize(gA, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
kS, dS = sift.detectAndCompute(small, None)
pairs = [(m, n) for m, n in cv2.BFMatcher().knnMatch(dS, dA, k=2) if m.distance < 0.8 * n.distance]
same = [m for m, n in pairs if np.hypot(*(np.array(kS[m.queryIdx].pt) / scale - kA[m.trainIdx].pt)) < 3]    # found again at the right place
sizes = [kA[m.trainIdx].size / kS[m.queryIdx].size for m in same]
print(f'shrunk ×{scale}: {len(kS):,} points · {len(same):,} found again at the right place · size ratio (median) {np.median(sizes):.2f} — 1/scale = {1 / scale:.2f}')

**What to notice:** at half size SIFT finds 1,622 points; 1,187 are found again at the right place, with sizes 2.00× smaller — each point carries the scale it was found at. 0.25 → 4.00 and 0.75 → 1.32 (1 ÷ 0.75 = 1.33). Slide 17's close-up gave 2.06 for a 2.23× zoom.

## C · Descriptors & matching

Describe each point by the gradients around it (**SIFT: 4 × 4 cells × 8 directions = 128 numbers**), match to the nearest descriptor, and keep only matches clearly better than the second best (**ratio test**).

In [ ]:
# C1 · One SIFT descriptor: 128 numbers (slide 22)
which = 0       # rank of the point by strength · try 50, 500

i = np.argsort([-p.response for p in kA])[which]; p, d = kA[i], dA[i]
x, y, r = int(p.pt[0]), int(p.pt[1]), int(p.size * 2)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.2), gridspec_kw={'width_ratios': [1, 4]})
ax[0].imshow(gA[max(0, y - r):y + r, max(0, x - r):x + r], cmap='gray'); ax[0].set_title(f'point {which}: size {p.size:.1f}, angle {p.angle:.0f}°'); ax[0].axis('off')
ax[1].bar(range(128), d, color=[plt.cm.tab20(c % 20) for c in np.repeat(range(16), 8)]); ax[1].set_title('16 cells × 8 directions (one colour per cell)'); plt.show()
print(len(d), 'numbers · largest', d.max(), '· cells with most gradient:', np.argsort(-d.reshape(16, 8).sum(1))[:4])

**What to notice:** 128 numbers = 4 × 4 cells × 8 directions (slide 22). Several bars stop at the same height (160 here): SIFT clips any one number at 0.2 of the vector's length and normalises again, so one strong edge cannot take over — one reason brighter light changes it so little (slide 21). Try 50 and 500: weaker points, different patterns.

In [ ]:
# C2 · Nearest descriptor + ratio test, judged with the true homography (slides 23–24)
ratio = 0.8     # keep if best ÷ second best < ratio · try 0.6, 1.01

knn = cv2.BFMatcher().knnMatch(dB, dA, k=2)                     # close-up → whole photo: best and second best
kept = [m for m, n in knn if m.distance < ratio * n.distance]
def right(m):                                                    # where H_REF sends the close-up point, within 5 px
    q = H_REF @ [*kB[m.queryIdx].pt, 1]; return np.hypot(*(q[:2] / q[2] - kA[m.trainIdx].pt)) < 5
ok = np.array([right(m) for m in kept])
print(f'ratio < {ratio}: {len(kept):,} matches kept · {ok.sum():,} right · {(~ok).sum():,} wrong ({(~ok).mean():.0%})')
canvas = np.zeros((max(gA.shape[0], gB.shape[0]), gA.shape[1] + gB.shape[1]), np.uint8); canvas[:gB.shape[0], :gB.shape[1]] = gB; canvas[:gA.shape[0], gB.shape[1]:] = gA
plt.figure(figsize=(14, 6)); plt.imshow(canvas, cmap='gray'); plt.axis('off')
for m, good in list(zip(kept, ok))[::max(1, len(kept) // 80)]:
    (xb, yb), (xa, ya) = kB[m.queryIdx].pt, kA[m.trainIdx].pt; plt.plot([xb, xa + gB.shape[1]], [yb, ya], c='lime' if good else 'red', lw=0.8)
plt.show()

**What to notice:** ratio 0.8 keeps 692 matches, 527 right and 165 wrong — slide 24 and the title slide. 0.6 keeps 383 with only 18 wrong (5%); 1.01 is no test at all: 3,048 matches, 648 right, 79% wrong. Every threshold leaves some wrong matches — part D deals with them.

## D · RANSAC

Least squares listens to every point, so a few wrong ones bend the answer. **RANSAC** fits a model to a tiny random sample, counts who agrees, and keeps the model with the most votes.

In [ ]:
# D1 · One wrong match ruins a least-squares line (slide 28)
n_wrong = 3     # · try 0, 6

rng = np.random.default_rng(3); x = np.sort(rng.uniform(0, 10, 20)); y = 0.6 * x + 2 + rng.normal(0, 0.15, 20)   # the slide's 20 good points
xw, yw = np.array([1.5, 2.5, 3.5, 6, 7.5, 9])[:n_wrong], np.array([8.5, 9, 8, 1, 0.5, 9.5])[:n_wrong]
slope_all = np.polyfit(np.r_[x, xw], np.r_[y, yw], 1)[0]; slope_good = np.polyfit(x, y, 1)[0]
plt.scatter(x, y, c='#855400'); plt.scatter(xw, yw, c='red'); t = np.array([0, 10]); plt.plot(t, np.polyval(np.polyfit(np.r_[x, xw], np.r_[y, yw], 1), t), 'r')
plt.plot(t, 0.6 * t + 2, 'k--'); plt.show()
print(f'{n_wrong} wrong points: slope with all {slope_all:.2f} · with the good ones only {slope_good:.2f} · true 0.6')

**What to notice:** three wrong points pull the slope from 0.57 to 0.36 — slide 28. With 0 the two fits agree (0.57; the true 0.6 is off only by the noise); with 6, all-points gives 0.29.

In [ ]:
# D2 · RANSAC for a line, by hand (slides 29–30)
tries = 20      # · try 5, 100

rng = np.random.default_rng(11); gx_ = rng.uniform(0, 10, 30); gy_ = 0.6 * gx_ + 2 + rng.normal(0, 0.15, 30)   # the demo's 50 points:
X = np.r_[gx_, rng.uniform(0, 10, 20)]; Y = np.r_[gy_, rng.uniform(0, 10, 20)]                                      # 30 on a line, 20 anywhere
pick = np.random.default_rng(0); best = (0, None)
for t in range(tries):
    i, j = pick.choice(50, 2, replace=False)
    a, b, c = Y[j] - Y[i], X[i] - X[j], X[j] * Y[i] - X[i] * Y[j]                # the line through the two points: ax + by + c = 0
    votes = np.abs(a * X + b * Y + c) / np.hypot(a, b) < 0.35
    if votes.sum() > best[0]: best = (votes.sum(), votes); print(f'try {t + 1:3d}: {votes.sum()} inliers — best so far')
slope, icpt = np.polyfit(X[best[1]], Y[best[1]], 1)                                # refit on the inliers
print(f'after {tries} tries: {best[0]} inliers · refit y = {slope:.2f}x + {icpt:.2f} (true 0.6x + 2)')

**What to notice:** try 2 already draws two good points and gets 31 inliers; try 18 finds 32, and the refit on them gives y = 0.58x + 2.10 against the true 0.6x + 2. Even 5 tries find the line here; D3 says 11 makes it 99% sure. 100 tries add nothing.

In [ ]:
# D3 · How many tries? N = ln(1 − p) / ln(1 − wˢ), then check it by simulation (slides 31–32)
w = 0.6     # share of good points · try 0.3, 0.9
s = 2       # points per sample (line 2, homography 4) · try 4, 8

p = 0.99
N = int(np.ceil(np.log(1 - p) / np.log(1 - w ** s)))
print(f'N = ln(1 − {p}) ÷ ln(1 − {w}^{s}) = {np.log(1 - p):.3f} ÷ {np.log(1 - w ** s):.3f} → {N} tries')
runs = np.random.default_rng(0).random((20000, N, s)) < w            # 20,000 RANSAC runs: is each drawn point good?
print(f'simulated: {(runs.all(2).any(1)).mean():.2%} of runs had at least one all-good sample in {N} tries (aim: {p:.0%}; 20,000 runs wobble by about ±0.1%)')

**What to notice:** w 0.6 and s 2 — slide 30's points — need 11 tries, as slide 31 derived, and 99.23% of the simulated runs succeed. w 0.3 with s 4 needs 567: fewer good points and bigger samples make N explode. w 0.9 with s 8 needs 9.

In [ ]:
# D4 · The photo pair: a homography with RANSAC (title slide and slide 33)
reproj = 4.0    # pixels: how close a point must land to vote · try 1.0, 10.0

good = [m for m, n in cv2.BFMatcher().knnMatch(dB, dA, k=2) if m.distance < 0.8 * n.distance]
src = np.float32([kB[m.queryIdx].pt for m in good]); dst = np.float32([kA[m.trainIdx].pt for m in good])
Hm, mask = cv2.findHomography(src, dst, cv2.RANSAC, reproj)
warped = cv2.warpPerspective(detail, Hm, full.shape[1::-1])
plt.figure(figsize=(7, 8)); plt.imshow(cv2.cvtColor(cv2.addWeighted(full, 0.5, warped, 0.5, 0), cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()
print(f'{len(good)} matches → {mask.sum()} inliers at {reproj} px · the close-up laid over the whole photo')

**What to notice:** 692 ratio-test matches → 527 inliers at 4 px — the lines on the title slide — and the close-up lands on its own window in the overlay. 1 px is too strict (381: good matches are a little off and vote no); 10 px keeps 527 here too.

In [ ]:
# D5 · A second view with a known answer: how far off is the homography? (slide 33)
angle = 20      # degrees the view is turned · try 0, 45

h, w_ = gA.shape; Ht = cv2.getRotationMatrix2D((w_ / 2, h / 2), angle, 0.8); Ht = np.vstack([Ht, [0.0002, 0.0001, 1]])   # turn, shrink, a little perspective
view = cv2.warpPerspective(gA, Ht, (w_, h))
kV, dV = sift.detectAndCompute(view, None)
good = [m for m, n in cv2.BFMatcher().knnMatch(dA, dV, k=2) if m.distance < 0.8 * n.distance]
src = np.float32([kA[m.queryIdx].pt for m in good]); dst = np.float32([kV[m.trainIdx].pt for m in good])
corners = np.float32([[0, 0], [w_, 0], [w_, h], [0, h]])[None]
err = lambda Hm: np.abs(cv2.perspectiveTransform(corners, Hm) - cv2.perspectiveTransform(corners, Ht)).max()
print(f'{len(good)} matches · corner error: RANSAC {err(cv2.findHomography(src, dst, cv2.RANSAC, 4.0)[0]):.1f} px · plain least squares {err(cv2.findHomography(src, dst, 0)[0]):.1f} px')

**What to notice:** turned 20°: corner error 0.2 px with RANSAC, 38.3 px with plain least squares on the same matches. 0°: 0.1 vs 34.4; 45°: 0.3 vs 50.0. The wrong matches pull least squares every time — slide 28 again, now in 2-D.

---
Back to the slides: [session 6 deck](https://classes.incortx.com/ComputerVision/session-06/slides.html) · exam-style questions: [practice bank](https://classes.incortx.com/ComputerVision/practice/#s6).